In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter   
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

C:\Users\akash kashyap\AppData\Local\Temp\ipykernel_17400\2544289455.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
C:\Users\akash kashyap\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
DATA_PATH = "./data"
DB_PATH = "./chroma_db"

In [3]:
def load_pdfs(data_path):
    all_docs = []
    for filename in os.listdir(data_path):
        if filename.lower().endswith(".pdf"):
            file_path = os.path.join(data_path, filename)
            print(f"Loading: {filename}")
            loader = PyPDFLoader(file_path)
            docs = loader.load()
            all_docs.extend(docs)
    return all_docs

docs = load_pdfs(DATA_PATH)
print(f"\nTotal pages loaded: {len(docs)}")

Loading: Ethics.pdf
Loading: IndianConstitution.pdf
Loading: Learning_SQL.pdf
Loading: ML Notes.pdf
Loading: PromptEng.pdf
Loading: ujwal_resume.pdf

Total pages loaded: 1098


In [4]:

print(docs[0].page_content[:500])
print(docs[0].metadata)

Ethics
{'producer': 'Adobe Acrobat Pro 10.1.2', 'creator': 'Adobe Acrobat Pro 10.1.2', 'creationdate': '2013-07-18T17:14:39+05:30', 'moddate': '2013-10-05T11:47:34+05:30', 'title': '', 'source': './data\\Ethics.pdf', 'total_pages': 178, 'page': 0, 'page_label': '1'}


In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(docs)
print(f"Total chunks created: {len(chunks)}")


Total chunks created: 3093


In [6]:
print(chunks[0].page_content[:300])
print(chunks[0].metadata)

Ethics
{'producer': 'Adobe Acrobat Pro 10.1.2', 'creator': 'Adobe Acrobat Pro 10.1.2', 'creationdate': '2013-07-18T17:14:39+05:30', 'moddate': '2013-10-05T11:47:34+05:30', 'title': '', 'source': './data\\Ethics.pdf', 'total_pages': 178, 'page': 0, 'page_label': '1'}


In [7]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=DB_PATH
)
vectorstore.persist()

print(f"Vector store saved at: {DB_PATH}")

C:\Users\akash kashyap\AppData\Local\Temp\ipykernel_17400\3269693850.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store saved at: ./chroma_db


C:\Users\akash kashyap\AppData\Local\Temp\ipykernel_17400\3269693850.py:10: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [8]:
query = "full?"

results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)


--- Result 1 ---
+------------+---------+---------------+
| product_cd | cust_id | avail_balance |
+------------+---------+---------------+
| CD         |       1 |       3000.00 |
| CD         |       6 |      10000.00 |
| CD         |       7 |       5000.00 |
| CD         |       9 |       1500.00 |
| CHK        |       1 |       1057.75 |
| CHK        |       2 |       2258.02 |
| CHK        |       3 |       1057.75 |
| CHK        |       4 |        534.12 |
| CHK        |       5 |       2237.97 |
| CHK        |       6 |        122.37 |
| CHK        |       8 |       3487.19 |
| CHK        |       9 |        125.67 |
| CHK        |      10 |      23575.12 |
| CHK        |      12 |      38552.05 |
| MM         |       3 |       2212.50 |
| MM         |       4 |       5487.09 |
| MM         |       9 |       9345.55 |
| SAV        |       1 |        500.00 |
| SAV        |       2 |        200.00 |
| SAV        |       4 |        767.77 |
| SAV        |       8 |        387.99 